<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/06_portfolio_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

# 07. Evaluación temporal de las reglas de cartera

## ¿Qué habría ocurrido al construir y mantener carteras con estas reglas, utilizando únicamente la información admisible en cada decisión?

En 06 construimos una cartera candidata de mínimo CVaR en un único corte histórico. Ese resultado describe su comportamiento bajo los escenarios utilizados para optimizarla, no su desempeño posterior.

En 07 reconstruiremos decisiones sucesivas y mediremos lo que ocurre después de cada una. Compararemos tres reglas: pesos iguales, mínimo CVaR con probabilidades exponenciales y mínimo CVaR condicionado por VIX.

Conservaremos los ocho ETFs y las restricciones de 06: cartera plenamente invertida, sin posiciones cortas, margen ni efectivo, y un tope de 15% por instrumento. Los escenarios de riesgo serán inicialmente de una sesión; el calendario de rebalanceo y la convención de ejecución se definirán por separado.

El ejercicio será retrospectivo. El universo se seleccionó después de observar la historia y la muestra ya fue explorada, por lo que no representa un *holdout* intacto ni resultados de operación real.

**Siguiente pregunta:** ¿Cuándo calculamos los pesos, cuándo podemos ejecutarlos y desde qué momento les atribuimos resultados?

## 2. ¿Cuándo decidimos y desde cuándo cuentan los resultados?

Todos los desplazamientos de esta convención corresponden a sesiones del calendario publicado, no a días naturales. Separamos la información que sirve para decidir de los retornos que atribuimos después a la decisión.

| Momento | Información o acción |
| --- | --- |
| Hasta el cierre de $t-1$ | Últimos retornos utilizados para ajustar el modelo. |
| Antes de ejecutar en $t$ | Calculamos los pesos con esa historia y el VIX de $t-1$. |
| Al cierre de $t$ | Ejecutamos hipotéticamente el rebalanceo. |
| Del cierre de $t$ al cierre de $t+1$ | Primer retorno atribuible a los pesos nuevos. |

Ejecutar al cierre es un supuesto del backtest, no una garantía de obtener ese precio en una operación real.

El retorno fechado $t$ pertenece a los pesos anteriores. En la primera inversión, la evaluación comienza después de ejecutar la cartera; todavía no calculamos resultados del periodo anterior a esa inversión.

No usamos el retorno ni el VIX de $t$ para decidir los pesos ejecutados al cierre de $t$.

Consideramos disponible el VIX de $t-1$ antes de ejecutar en $t$, siguiendo la evidencia aportada por el proveedor: normalmente estaría disponible hacia las 22:00 US/Eastern. Esta es una convención retrospectiva basada en esa comunicación, no una disponibilidad histórica verificada para cada fecha ni una garantía de publicación.

El VIX tiene dos usos distintos. En la muestra histórica de ajuste, cada retorno fechado $x$ conserva su asociación con el VIX de $x-2$, según la convención de 04/05. El VIX de $t-1$ identifica el estado conocido para la decisión actual. No cambiaremos la alineación histórica para hacer coincidir ambas fechas.

Las tres reglas —pesos iguales, mínimo CVaR exponencial y mínimo CVaR condicionado por VIX— compartirán la misma convención de decisión, ejecución y atribución de resultados.

Como ejemplo conceptual, considere tres sesiones consecutivas. En la sesión anterior termina la información utilizada; en la sesión de ejecución aplicamos los nuevos pesos al cierre; en la sesión siguiente observamos el primer retorno de los nuevos pesos. Este ejemplo no presupone precios, pesos ni rendimientos concretos.

**Siguiente pregunta:** ¿En qué sesiones rebalancearemos y cuánta historia exigiremos antes de invertir?

## 3. ¿Cuándo rebalanceamos y cuánta historia necesitamos?

### Frecuencia mensual

Para las tres reglas, el rebalanceo se ejecutará hipotéticamente al cierre de la última sesión publicada de cada mes calendario completo. La decisión utiliza información hasta la sesión anterior a la ejecución, según la convención temporal anterior. Identificaremos esas fechas con el calendario publicado, sin suponer que todos los meses terminan en un día hábil. El último día disponible del corte no se tratará automáticamente como cierre mensual: octubre de 2026 es parcial.

La frecuencia mensual es una decisión operativa inicial, no una frecuencia demostrada como óptima.

### Historia mínima

Exigiremos al menos 504 observaciones históricas alineadas y admisibles para el modelo condicionado por VIX. El mínimo CVaR exponencial utilizará exactamente esas mismas fechas de ajuste. Las 504 observaciones son filas conjuntas de retornos y estado histórico asociado, no días naturales.

La ventana será expansiva: comienza en la primera observación alineada disponible y crece en cada rebalanceo. La primera inversión ocurrirá en el primer cierre mensual que cumpla este requisito; todavía no calculamos cuál es esa fecha. Las 504 observaciones son un requisito de diseño, no una garantía de precisión.

### Parámetros fijos

Mantendremos una semivida exponencial de 1,102 sesiones, CVaR al 95% y un horizonte de riesgo de una sesión, junto con los ocho ETFs y las restricciones de 06. En cada rebalanceo condicionado reconstruiremos los cortes de VIX y los objetivos de *entropy pooling* con la muestra admisible de esa fecha. No reutilizaremos cortes ni probabilidades ajustadas con toda la historia de 04, ni buscaremos parámetros retrospectivamente para mejorar resultados.

Optimizar riesgo de una sesión y rebalancear mensualmente son decisiones distintas: no estamos minimizando directamente el CVaR mensual.

### Entre rebalanceos

No restauraremos diariamente los pesos objetivo. Los pesos evolucionarán con los retornos de los instrumentos. Pesos iguales significa volver a 12.5% por ETF en cada rebalanceo, no mantener 12.5% todos los días.

El máximo de 15% se exigirá sobre los pesos objetivo al rebalancear. La deriva posterior puede superar ese nivel sin activar operaciones adicionales. Tras el último rebalanceo, mantendremos las posiciones hasta el corte disponible, aunque el periodo final sea parcial.

**Siguiente pregunta:** ¿Podemos reconstruir estas fechas y sus muestras admisibles sin utilizar información futura?

## 4. ¿Qué fechas podemos usar para rebalancear?

Construimos los cierres mensuales con el calendario publicado y, para cada uno, identificamos la muestra expansiva disponible hasta la sesión anterior. Ambos modelos optimizados compartirán esas filas. El último mes del corte, octubre de 2026, es parcial y queda en la auditoría como no elegible. El horizonte de riesgo es una sesión y la semivida, 1,102 sesiones.

El raw VIX se verifica con el hash de su captura y la alineación con el hash de su manifiesto. Los hashes de los dos Parquet de matrices se comparan con los registrados posteriormente al crear la alineación: el manifiesto original del corte no ofrece hashes históricos de referencia para ellos. El catálogo tampoco tiene aquí un hash de referencia; verificamos sus identificadores y estados. La disponibilidad histórica del VIX sigue siendo una convención retrospectiva, no una comprobación point-in-time.

In [1]:
from pathlib import Path
from datetime import date
import hashlib, json, site
import numpy as np

def project_root():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'scenario-portfolio-lab', parent / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if (candidate / 'config' / 'universe.csv').is_file() and (candidate / 'src' / 'scenario_portfolio_lab').is_dir():
                return candidate
    raise RuntimeError('Ejecute la notebook dentro del repositorio completo scenario-portfolio-lab.')

ROOT = project_root()
project_site = ROOT / '.venv' / 'Lib' / 'site-packages'
if project_site.is_dir():
    site.addsitedir(str(project_site))
import polars as pl

RUN = ROOT / 'data/published/cut_2026-10-09_run_2026-10-11_full_window_final'
ALIGNMENT = ROOT / 'data/published/vix_state_alignment_2026-10-31'
VIX_CAPTURE = ROOT / 'data/raw/vix_index_access_2026-10-29'
price_path = RUN / 'gold/adjusted_close_matrix.parquet'
return_path = RUN / 'gold/eligible_simple_total_return_matrix.parquet'
catalog_path = RUN / 'reports/eligible_universe.csv'
audit_path = ALIGNMENT / 'vix_return_alignment.parquet'
vix_path = VIX_CAPTURE / 'raw/VIX_INDEX_DATA_daily.json'
run_manifest = json.loads((RUN / 'run_manifest.json').read_text(encoding='utf-8'))
alignment_manifest = json.loads((ALIGNMENT / 'alignment_manifest.json').read_text(encoding='utf-8'))
vix_manifest = json.loads((VIX_CAPTURE / 'manifest.json').read_text(encoding='utf-8'))
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
integrity = {
    'VIX raw: hash de captura': sha256(vix_path) == vix_manifest['raw_sha256'] == alignment_manifest['inputs']['vix_raw']['sha256_capture_manifest_reference'],
    'Alineación: hash de su manifiesto': sha256(audit_path) == alignment_manifest['outputs']['alignment_table']['sha256'],
    'Precios: hash posterior de alineación': sha256(price_path) == alignment_manifest['inputs']['price_calendar']['sha256_computed_now'],
    'Retornos: hash posterior de alineación': sha256(return_path) == alignment_manifest['inputs']['simple_return_matrix']['sha256_computed_now'],
}
if not all(integrity.values()):
    raise AssertionError([name for name, ok in integrity.items() if not ok])

# Validamos los esquemas reales antes de seleccionar las columnas de trabajo.
price_schema = pl.read_parquet_schema(price_path); return_schema = pl.read_parquet_schema(return_path)
audit_schema = pl.read_parquet_schema(audit_path)
catalog = pl.read_csv(catalog_path)
required_audit = {'return_row_id', 'return_date', 'return_start_date', 'vix_source_date', 'vix_close', 'alignment_status'}
if list(price_schema)[0] != 'session_date' or list(return_schema)[0] != 'session_date' or not required_audit.issubset(audit_schema) or not {'symbol', 'asset_class', 'universe_status'}.issubset(catalog.columns):
    raise AssertionError('Esquema de entrada inesperado; revisar archivos de origen.')
eligible = catalog.filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible['symbol'].to_list()
SYMBOLS = ['SPY', 'VEA', 'EEM', 'TLT', 'BIL', 'GLD', 'USO', 'VNQ']
if len(eligible_symbols) != 125 or len(set(eligible_symbols)) != 125 or list(return_schema)[1:] != eligible_symbols:
    raise AssertionError('El catálogo elegible no coincide exactamente con las columnas de retornos.')
if any(eligible_symbols.count(symbol) != 1 or symbol not in price_schema for symbol in SYMBOLS):
    raise AssertionError('Falta o se repite un ETF en los insumos publicados.')
prices = pl.read_parquet(price_path, columns=['session_date', *SYMBOLS])
returns = pl.read_parquet(return_path, columns=['session_date', *SYMBOLS])
audit = pl.read_parquet(audit_path)
calendar = prices['session_date'].to_list()
cut = date.fromisoformat(run_manifest['approved_period']['end'])
start = date.fromisoformat(run_manifest['approved_period']['start'])
raw_vix = json.loads(vix_path.read_text(encoding='utf-8'))
if raw_vix.get('symbol') != 'VIX' or raw_vix.get('interval') != 'daily':
    raise AssertionError('El raw no identifica la serie VIX diaria.')
vix_records = [(date.fromisoformat(row['date']), float(row['close'])) for row in raw_vix['data']]
if len({d for d, _ in vix_records}) != len(vix_records):
    raise AssertionError('El raw VIX contiene fechas duplicadas.')
# No usamos los registros VIX posteriores al corte, aunque se conservan en el raw original.
vix_by_date = {d: value for d, value in vix_records if d <= cut}
if len(calendar) != 2205 or calendar[0] != start or calendar[-1] != cut or returns['session_date'].to_list() != calendar:
    raise AssertionError('El calendario de matrices no coincide con la ventana publicada.')
if calendar != sorted(calendar) or len(set(calendar)) != len(calendar):
    raise AssertionError('Las fechas del calendario no son únicas y crecientes.')
if returns.head(1).select(SYMBOLS).null_count().row(0) != (1,) * len(SYMBOLS):
    raise AssertionError('La primera fila de retornos debe ser nula por ausencia de precio anterior.')
return_values = returns.slice(1).select(SYMBOLS).to_numpy()
price_values = prices.select(SYMBOLS).to_numpy()
if not np.isfinite(return_values).all() or np.any(return_values <= -1) or not np.isfinite(price_values).all() or np.any(price_values <= 0):
    raise AssertionError('Hay retornos o precios inválidos en los ocho ETF.')

audit_rows = audit.to_dicts()
if len(audit_rows) != 2204 or [row['return_date'] for row in audit_rows] != calendar[1:]:
    raise AssertionError('La auditoría no conserva todas las fechas de retorno en orden.')
if audit_rows[0]['alignment_status'] != 'UNALIGNABLE_PRE_PUBLISHED_CALENDAR_START' or any(row['alignment_status'] != 'ALIGNED' for row in audit_rows[1:]):
    raise AssertionError('Estados de alineación inesperados; no se descartan filas silenciosamente.')
for i, row in enumerate(audit_rows):
    if row['return_row_id'] != i or row['return_start_date'] != calendar[i]:
        raise AssertionError(f'Identidad o inicio de retorno incorrecto: {row}')
    if i >= 1 and (row['vix_source_date'] != calendar[i-1] or row['vix_source_date'] not in vix_by_date or row['vix_close'] != vix_by_date[row['vix_source_date']] or not np.isfinite(row['vix_close']) or row['vix_close'] <= 0):
        raise AssertionError(f'Alineación histórica VIX x-2 inválida: {row}')
aligned_rows = audit_rows[1:]
aligned_dates = [row['return_date'] for row in aligned_rows]

# Un mes queda completo sólo si el calendario publicado alcanza el mes siguiente; octubre final es parcial.
month_ends = [i for i, day in enumerate(calendar) if i == len(calendar)-1 or (calendar[i+1].year, calendar[i+1].month) != (day.year, day.month)]
M, TAU, RISK_HORIZON = 504, 1102, 1
rebalance_rows = []
for i in month_ends:
    execution = calendar[i]; previous = calendar[i-1] if i > 0 else None
    fit = [d for d in aligned_dates if previous is not None and d <= previous]
    decision_vix = vix_by_date.get(previous)
    next_return = calendar[i+1] if i+1 < len(calendar) else None
    reasons = []
    if i == len(calendar)-1: reasons.append('MES_FINAL_PARCIAL')
    if len(fit) < M: reasons.append('HISTORIA_ALINEADA_INSUFICIENTE')
    if decision_vix is None or not np.isfinite(decision_vix) or decision_vix <= 0: reasons.append('VIX_DECISIÓN_NO_DISPONIBLE')
    if next_return is None: reasons.append('SIN_SESIÓN_POSTERIOR_EN_EL_CORTE')
    rebalance_rows.append({'fecha_ejecución': execution, 'fin_información': previous, 'inicio_ajuste': fit[0] if fit else None, 'fin_ajuste': fit[-1] if fit else None, 'observaciones': len(fit), 'fecha_vix_decisión': previous, 'vix_decisión': decision_vix, 'primer_retorno_nuevo': next_return, 'elegible': not reasons, 'motivo': ' | '.join(reasons) if reasons else 'ELEGIBLE'})
rebalance_audit = pl.DataFrame(rebalance_rows)
eligible_rebalances = rebalance_audit.filter(pl.col('elegible'))

# Comprobamos identidad, fechas y ventanas mediante posiciones, sin imputar ni ajustar filas.
positions = {day: i for i, day in enumerate(calendar)}
eligible_rows = eligible_rebalances.to_dicts()
reason_counts = {}
for row in rebalance_rows:
    if not row['elegible']:
        for reason in row['motivo'].split(' | '): reason_counts[reason] = reason_counts.get(reason, 0) + 1
controls = {**integrity,
    'Fechas únicas y crecientes': calendar == sorted(set(calendar)) and aligned_dates == sorted(set(aligned_dates)),
    'Ocho ETF en orden y elegibles': returns.columns == ['session_date', *SYMBOLS] and prices.columns == ['session_date', *SYMBOLS] and all(eligible_symbols.count(s) == 1 for s in SYMBOLS),
    'Cierres completos; octubre parcial excluido': all(positions[r['fecha_ejecución']] in month_ends[:-1] for r in eligible_rows) and rebalance_rows[-1]['motivo'].startswith('MES_FINAL_PARCIAL') and (cut.year, cut.month) == (2026, 10),
    'Ajuste anterior y conteo directo': all(r['fin_ajuste'] <= r['fin_información'] < r['fecha_ejecución'] and r['observaciones'] == sum(d <= r['fin_información'] for d in aligned_dates) for r in eligible_rows),
    'Mínimo y ventanas expansivas anidadas': all(r['observaciones'] >= M and r['inicio_ajuste'] == aligned_dates[0] for r in eligible_rows) and all(a['observaciones'] < b['observaciones'] for a, b in zip(eligible_rows, eligible_rows[1:])),
    'Alineación histórica x-2 y raw exacto': len(aligned_rows) == 2203 and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] and r['vix_close'] == vix_by_date[r['vix_source_date']] for r in aligned_rows),
    'VIX decisión en sesión anterior': all(r['fecha_vix_decisión'] == calendar[positions[r['fecha_ejecución']]-1] and r['vix_decisión'] == vix_by_date[r['fecha_vix_decisión']] for r in eligible_rows),
    'Primer retorno es sesión siguiente': all(r['primer_retorno_nuevo'] == calendar[positions[r['fecha_ejecución']]+1] and r['primer_retorno_nuevo'] > r['fecha_ejecución'] for r in eligible_rows),
    'Sin insumos posteriores al corte': all(d <= cut for d in calendar) and all(r['vix_source_date'] <= cut for r in aligned_rows) and all(r['fin_información'] <= cut for r in eligible_rows),
}
if not all(controls.values()):
    raise AssertionError([name for name, ok in controls.items() if not ok])
display(pl.DataFrame({'símbolo': SYMBOLS, 'clase': [eligible.filter(pl.col('symbol') == s)['asset_class'][0] for s in SYMBOLS]}))
print(f'Sesiones: {len(calendar)}; retornos: {len(calendar)-1}; filas alineadas: {len(aligned_rows)}; ETF: {len(SYMBOLS)}; M={M}; semivida={TAU}; horizonte={RISK_HORIZON}.')
print(f'Rebalanceos elegibles: {eligible_rebalances.height}; primer y último grupo de cinco:')
display(eligible_rebalances.head(5)); display(eligible_rebalances.tail(5))
display(pl.DataFrame({'motivo': list(reason_counts), 'conteo': list(reason_counts.values())}))
display(pl.DataFrame({'control': list(controls), 'resultado': ['PASS' if ok else 'FAIL' for ok in controls.values()]}))


símbolo,clase
str,str
"""SPY""","""etf"""
"""VEA""","""etf"""
"""EEM""","""etf"""
"""TLT""","""etf"""
"""BIL""","""etf"""
"""GLD""","""etf"""
"""USO""","""etf"""
"""VNQ""","""etf"""


Sesiones: 2205; retornos: 2204; filas alineadas: 2203; ETF: 8; M=504; semivida=1102; horizonte=1.
Rebalanceos elegibles: 81; primer y último grupo de cinco:


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2020-01-31,2020-01-30,2018-01-04,2020-01-30,521,2020-01-30,15.49,2020-02-03,true,"""ELEGIBLE"""
2020-02-28,2020-02-27,2018-01-04,2020-02-27,540,2020-02-27,39.16,2020-03-02,true,"""ELEGIBLE"""
2020-03-31,2020-03-30,2018-01-04,2020-03-30,562,2020-03-30,57.08,2020-04-01,true,"""ELEGIBLE"""
2020-04-30,2020-04-29,2018-01-04,2020-04-29,583,2020-04-29,31.23,2020-05-01,true,"""ELEGIBLE"""
2020-05-29,2020-05-28,2018-01-04,2020-05-28,603,2020-05-28,28.59,2020-06-01,true,"""ELEGIBLE"""


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2026-05-29,2026-05-28,2018-01-04,2026-05-28,2110,2026-05-28,15.74,2026-06-01,true,"""ELEGIBLE"""
2026-06-30,2026-06-29,2018-01-04,2026-06-29,2131,2026-06-29,17.65,2026-07-01,true,"""ELEGIBLE"""
2026-07-31,2026-07-30,2018-01-04,2026-07-30,2153,2026-07-30,17.09,2026-08-03,true,"""ELEGIBLE"""
2026-08-31,2026-08-28,2018-01-04,2026-08-28,2174,2026-08-28,14.43,2026-09-01,true,"""ELEGIBLE"""
2026-09-30,2026-09-29,2018-01-04,2026-09-29,2195,2026-09-29,16.04,2026-10-01,true,"""ELEGIBLE"""


motivo,conteo
str,i64
"""HISTORIA_ALINEADA_INSUFICIENTE""",24
"""MES_FINAL_PARCIAL""",1
"""SIN_SESIÓN_POSTERIOR_EN_EL_COR…",1


control,resultado
str,str
"""VIX raw: hash de captura""","""PASS"""
"""Alineación: hash de su manifie…","""PASS"""
"""Precios: hash posterior de ali…","""PASS"""
"""Retornos: hash posterior de al…","""PASS"""
"""Fechas únicas y crecientes""","""PASS"""
…,…
"""Mínimo y ventanas expansivas a…","""PASS"""
"""Alineación histórica x-2 y raw…","""PASS"""
"""VIX decisión en sesión anterio…","""PASS"""


## 5. ¿Qué carteras obtendríamos en el primer rebalanceo?

Con la primera fecha elegible reconstruimos la misma muestra admisible para las tres carteras: pesos iguales y mínimo CVaR al 95% bajo probabilidades exponenciales o condicionadas por el VIX previo. El ajuste condicionado estima sus cortes y momentos sólo con esa historia. Comparamos cada cartera bajo ambas distribuciones de ajuste; esta comparación describe los escenarios empleados, no resultados posteriores.

Las optimizaciones usan retornos simples conjuntos a una sesión, inversión completa y pesos entre 0% y 15%. La referencia exponencial tiene semivida de 1,102 sesiones. Para entropy pooling usamos residuos estandarizados: tolerancias de igualdad y factibilidad $10^{-9}$, KKT $10^{-8}$ y normalización $10^{-12}$. Para la programación lineal usamos $10^{-9}$ en factibilidad de pesos y $10^{-8}$ en conciliación del objetivo y comparación de CVaR; estos valores están en unidades decimales de retorno.

In [2]:
from scipy.optimize import linprog, minimize, root, least_squares
from scipy.special import logsumexp
from scipy.sparse import hstack, csr_matrix, eye
import plotly.graph_objects as go

first_rebalance = eligible_rebalances.row(0, named=True)
execution_date = first_rebalance['fecha_ejecución']
fit_end = first_rebalance['fin_información']
fit_audit = audit.filter((pl.col('alignment_status') == 'ALIGNED') & (pl.col('return_date') <= fit_end))
fit_dates = fit_audit['return_date'].to_list()
fit_returns_frame = returns.filter(pl.col('session_date').is_in(fit_dates))
fit_returns = fit_returns_frame.select(SYMBOLS).to_numpy().astype(float)
fit_vix = fit_audit['vix_close'].to_numpy().astype(float)
if not (execution_date == date(2020, 1, 31) and fit_end == date(2020, 1, 30) and len(fit_dates) == first_rebalance['observaciones'] == 521 and fit_returns_frame['session_date'].to_list() == fit_dates and fit_dates[0] == first_rebalance['inicio_ajuste'] and fit_dates[-1] == first_rebalance['fin_ajuste'] and fit_returns.shape == (521, 8)):
    raise AssertionError('La muestra del primer rebalanceo no coincide con el calendario.')

# Las edades se miden en posiciones del calendario de sesiones, respecto de la última fila de ajuste.
fit_positions = np.array([positions[d] for d in fit_dates])
fit_ages = fit_positions[-1] - fit_positions
p_exp = np.exp(-np.log(2.0) * fit_ages / TAU); p_exp /= p_exp.sum()
sorted_vix = np.sort(fit_vix)
vix_p25 = float(sorted_vix[int(np.ceil(len(fit_vix) * 0.25)) - 1])
vix_p75 = float(sorted_vix[int(np.ceil(len(fit_vix) * 0.75)) - 1])
if not vix_p25 < vix_p75: raise AssertionError('Los cortes VIX no son distintos.')
fit_groups = np.where(fit_vix <= vix_p25, 1, np.where(fit_vix <= vix_p75, 2, 3))
decision_vix = float(first_rebalance['vix_decisión'])
selected_group = 1 if decision_vix <= vix_p25 else (2 if decision_vix <= vix_p75 else 3)
group_values = fit_vix[fit_groups == selected_group]
if len(group_values) == 0: raise AssertionError('El grupo VIX seleccionado está vacío.')
target_mean = float(group_values.mean()); target_variance = float(np.mean((group_values - target_mean)**2))
if target_variance <= 0: raise AssertionError('La dispersión VIX crisp debe ser positiva.')
target_std = np.sqrt(target_variance); target_second = target_mean**2 + target_variance
a = (fit_vix - target_mean) / target_std; b = a**2; log_p = np.log(p_exp)

# Dual convexo: primero probamos la solución con eta=0; si no es factible, resolvemos la restricción activa.
def entropy_dual(theta):
    log_unscaled = log_p - theta[0] * a - theta[1] * b
    log_q = log_unscaled - logsumexp(log_unscaled); q = np.exp(log_q)
    return float(logsumexp(log_unscaled) + theta[1]), np.array([-q @ a, 1.0 - q @ b]), q, log_q

def entropy_hessian(theta):
    q = entropy_dual(theta)[2]; features = np.column_stack((a, b)); centered = features - q @ features
    return (centered * q[:, None]).T @ centered

boundary = root(lambda x: [entropy_dual(np.array([x[0], 0.0]))[1][0]], [0.0], method='hybr', options={'xtol': 1e-12})
if not boundary.success: raise RuntimeError(f'Entropy pooling, frontera: {boundary.message}')
boundary_theta = np.array([boundary.x[0], 0.0])
if entropy_dual(boundary_theta)[1][1] >= -1e-9:
    theta = boundary_theta; ep_case = 'INACTIVA'
else:
    ep_result = minimize(lambda x: entropy_dual(x)[0], [boundary_theta[0], 0.1], jac=lambda x: entropy_dual(x)[1], method='SLSQP', bounds=[(None, None), (0.0, None)], options={'ftol': 1e-12, 'maxiter': 10000})
    if not ep_result.success: raise RuntimeError(f'Entropy pooling, grupo {selected_group}: {ep_result.message}')
    polished = root(lambda x: entropy_dual(x)[1], ep_result.x, jac=entropy_hessian, method='hybr', options={'xtol': 1e-12})
    theta = polished.x
    if not polished.success or theta[1] <= 0 or np.max(np.abs(entropy_dual(theta)[1])) > 1e-8:
        refined = least_squares(lambda x: entropy_dual(x)[1], ep_result.x, bounds=([-np.inf, 0.0], [np.inf, np.inf]), xtol=1e-14, ftol=1e-14, gtol=1e-14, max_nfev=10000)
        if not refined.success: raise RuntimeError(f'Entropy pooling, pulido: {refined.message}')
        theta = refined.x
    if theta[1] <= 0: raise AssertionError('El multiplicador activo debe ser positivo.')
    ep_case = 'ACTIVA'
_, gradient, q_vix, log_q_vix = entropy_dual(theta)
eta = float(theta[1]); slack = float(gradient[1]); kl = float(q_vix @ (log_q_vix - log_p))
fd_point = np.array([0.17, 0.23]); fd_step = 1e-6
fd_gradient = np.array([(entropy_dual(fd_point + np.eye(2)[i] * fd_step)[0] - entropy_dual(fd_point - np.eye(2)[i] * fd_step)[0]) / (2 * fd_step) for i in range(2)])
gradient_error = float(np.max(np.abs(fd_gradient - entropy_dual(fd_point)[1])))

# CVaR: min z + sum(p*u)/0.05; u >= -(R@w)-z, u >= 0 y z libre.
scenario_count, asset_count = fit_returns.shape; tail = 0.05; cap = 0.15
constraint_matrix = hstack([csr_matrix(-fit_returns), csr_matrix(-np.ones((scenario_count, 1))), -eye(scenario_count, format='csr')], format='csr')
investment_matrix = csr_matrix(np.r_[np.ones(asset_count), np.zeros(1 + scenario_count)].reshape(1, -1))
bounds = [(0.0, cap)] * asset_count + [(None, None)] + [(0.0, None)] * scenario_count
def solve_cvar(probabilities):
    objective = np.r_[np.zeros(asset_count), 1.0, probabilities / tail]
    result = linprog(objective, A_ub=constraint_matrix, b_ub=np.zeros(scenario_count), A_eq=investment_matrix, b_eq=[1.0], bounds=bounds, method='highs')
    if not result.success: raise RuntimeError(f'CVaR HiGHS: {result.message}')
    return result

exp_solution = solve_cvar(p_exp); vix_solution = solve_cvar(q_vix)
portfolios = {'Pesos iguales': np.full(asset_count, 1.0 / asset_count), 'Mínimo CVaR exponencial': exp_solution.x[:asset_count], 'Mínimo CVaR VIX': vix_solution.x[:asset_count]}

# Ordenamos pérdidas y tomamos exactamente el 5% de masa; conciliamos con la identidad basada en VaR.
def tail_metrics(losses, probabilities):
    order = np.argsort(losses, kind='stable'); cumulative = np.cumsum(probabilities[order])
    var = float(losses[order[np.searchsorted(cumulative, 1.0 - tail, side='left')]])
    reverse = np.argsort(-losses, kind='stable'); p_sorted = probabilities[reverse]
    before = np.r_[0.0, np.cumsum(p_sorted[:-1])]
    included = np.minimum(np.maximum(tail - before, 0.0), p_sorted)
    cvar = float(included @ losses[reverse] / tail)
    independent = float(var + probabilities @ np.maximum(losses - var, 0.0) / tail)
    return cvar, independent, float(included.sum())

distributions = {'Exponencial': p_exp, 'Condicionada VIX': q_vix}
risk = {(name, distribution): tail_metrics(-(fit_returns @ weights), probabilities) for name, weights in portfolios.items() for distribution, probabilities in distributions.items()}
weight_table = pl.DataFrame({'ETF': SYMBOLS, **{name: 100 * weights for name, weights in portfolios.items()}})
risk_table = pl.DataFrame({'Cartera': list(portfolios), 'CVaR exponencial (%)': [100 * risk[(name, 'Exponencial')][0] for name in portfolios], 'CVaR condicionado VIX (%)': [100 * risk[(name, 'Condicionada VIX')][0] for name in portfolios]})

TOL = 1e-12; WEIGHT_TOL = 1e-9; CVAR_TOL = 1e-8
block5_controls = {
    'Muestra y alineación comunes': fit_returns_frame['session_date'].to_list() == fit_dates and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] and r['return_date'] <= fit_end for r in fit_audit.to_dicts()) and first_rebalance['fecha_vix_decisión'] == fit_end and decision_vix == vix_by_date[fit_end],
    'Cortes y grupos crisp': len(group_values) > 0 and np.all(np.isin(fit_groups, [1, 2, 3])) and fit_ages[-1] == 0 and np.all(np.diff(fit_ages) > 0) == False and np.all(np.diff(fit_ages) < 0),
    'Probabilidades normalizadas': all(np.isfinite(p).all() and np.all(p >= 0) and np.isclose(p.sum(), 1.0, atol=TOL, rtol=TOL) for p in distributions.values()),
    'Entropy pooling factible': abs(float(q_vix @ a)) <= 1e-9 and float(q_vix @ b) <= 1 + 1e-9 and np.isclose(q_vix @ fit_vix, target_mean, atol=1e-9 * target_std, rtol=0) and q_vix @ (fit_vix**2) <= target_second + 1e-9 * target_variance,
    'Entropy pooling KKT y gradiente': abs(gradient[0]) <= 1e-8 and eta >= 0 and slack >= -1e-8 and abs(eta * slack) <= 1e-8 and gradient_error <= 1e-7 and np.isfinite(kl) and kl >= -1e-12,
    'Pesos factibles sin corrección': all(np.isclose(w.sum(), 1, atol=WEIGHT_TOL, rtol=0) and np.all(w >= -WEIGHT_TOL) and np.all(w <= cap + WEIGHT_TOL) for w in portfolios.values()),
    'Cola exacta y CVaR independiente': all(np.isclose(mass, tail, atol=TOL, rtol=TOL) and np.isclose(cvar, independent, atol=TOL, rtol=TOL) for cvar, independent, mass in risk.values()),
    'Objetivos HiGHS concilian': abs(exp_solution.fun - risk[('Mínimo CVaR exponencial', 'Exponencial')][0]) <= CVAR_TOL and abs(vix_solution.fun - risk[('Mínimo CVaR VIX', 'Condicionada VIX')][0]) <= CVAR_TOL,
    'Óptimos no peores en su distribución': all(risk[('Mínimo CVaR exponencial', 'Exponencial')][0] <= risk[(name, 'Exponencial')][0] + CVAR_TOL and risk[('Mínimo CVaR VIX', 'Condicionada VIX')][0] <= risk[(name, 'Condicionada VIX')][0] + CVAR_TOL for name in portfolios),
    'Restricciones auxiliares de ambos LP': all(np.all(sol.x[asset_count+1:] >= -(fit_returns @ sol.x[:asset_count]) - sol.x[asset_count] - WEIGHT_TOL) and np.all(sol.x[asset_count+1:] >= -WEIGHT_TOL) for sol in (exp_solution, vix_solution)),
}
if not all(block5_controls.values()): raise AssertionError([name for name, ok in block5_controls.items() if not ok])

print(f'Rebalanceo: {execution_date}; ajuste: {fit_dates[0]} a {fit_end}; observaciones: {len(fit_dates)}.')
display(weight_table)
display(pl.DataFrame({'p25': [vix_p25], 'p75': [vix_p75], 'fecha VIX': [fit_end], 'VIX decisión': [decision_vix], 'grupo': [selected_group], 'n crisp': [len(group_values)], 'media objetivo': [target_mean], 'media obtenida': [float(q_vix @ fit_vix)], 'varianza límite': [target_variance], 'varianza obtenida': [float(q_vix @ ((fit_vix-target_mean)**2))], 'caso': [ep_case], 'eta': [eta], 'holgura': [slack], 'N_eff exp': [float(1 / (p_exp @ p_exp))], 'N_eff VIX': [float(1 / (q_vix @ q_vix))], 'KL': [kl], 'underflow': [int(np.count_nonzero(q_vix == 0))]}))
display(risk_table)
figure = go.Figure()
for name, weights in portfolios.items():
    figure.add_bar(name=name, y=SYMBOLS, x=100 * weights, orientation='h', hovertemplate='%{y}: %{x:.5f}%<extra>%{fullData.name}</extra>')
figure.add_vline(x=15, line_dash='dash', line_color='#444444')
figure.update_layout(title='Pesos objetivo del primer rebalanceo', barmode='group', paper_bgcolor='#FFF8E7', plot_bgcolor='#FFF8E7', font_color='#2D2A26', legend=dict(x=1.02, y=1), margin=dict(l=70, r=220, t=60, b=55))
figure.update_xaxes(title='Peso objetivo (%)', range=[0, 17], gridcolor='#E7DDCC'); figure.update_yaxes(categoryorder='array', categoryarray=list(reversed(SYMBOLS)))
figure.show()
display(pl.DataFrame({'Control': list(block5_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in block5_controls.values()]}))
print(f'Residuos KKT: igualdad={gradient[0]:.3e}, holgura={slack:.3e}, complementariedad={eta*slack:.3e}; error gradiente={gradient_error:.3e}.')


Rebalanceo: 2020-01-31; ajuste: 2018-01-04 a 2020-01-30; observaciones: 521.


ETF,Pesos iguales,Mínimo CVaR exponencial,Mínimo CVaR VIX
str,f64,f64,f64
"""SPY""",12.5,15.0,15.0
"""VEA""",12.5,15.0,15.0
"""EEM""",12.5,4.450925,0.666044
"""TLT""",12.5,15.0,15.0
"""BIL""",12.5,15.0,15.0
"""GLD""",12.5,15.0,15.0
"""USO""",12.5,5.549075,9.333956
"""VNQ""",12.5,15.0,15.0


p25,p75,fecha VIX,VIX decisión,grupo,n crisp,media objetivo,media obtenida,varianza límite,varianza obtenida,caso,eta,holgura,N_eff exp,N_eff VIX,KL,underflow
f64,f64,date,f64,i64,i64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64
12.88,17.97,2020-01-30,15.49,2,260,15.031346,15.031346,2.045285,2.045285,"""ACTIVA""",0.457661,-2.2204e-16,516.387041,272.274201,0.480215,0


Cartera,CVaR exponencial (%),CVaR condicionado VIX (%)
str,f64,f64
"""Pesos iguales""",1.194557,1.088444
"""Mínimo CVaR exponencial""",0.948994,0.855785
"""Mínimo CVaR VIX""",0.958341,0.85095


Control,Resultado
str,str
"""Muestra y alineación comunes""","""PASS"""
"""Cortes y grupos crisp""","""PASS"""
"""Probabilidades normalizadas""","""PASS"""
"""Entropy pooling factible""","""PASS"""
"""Entropy pooling KKT y gradient…","""PASS"""
"""Pesos factibles sin corrección""","""PASS"""
"""Cola exacta y CVaR independien…","""PASS"""
"""Objetivos HiGHS concilian""","""PASS"""
"""Óptimos no peores en su distri…","""PASS"""


Residuos KKT: igualdad=1.561e-17, holgura=-2.220e-16, complementariedad=-1.016e-16; error gradiente=3.218e-10.


## 6. ¿Qué ocurre después de ejecutar las primeras carteras?

Seguimos cada cartera desde su ejecución inicial hasta el cierre del siguiente rebalanceo elegible, incluido. Durante ese intervalo mantenemos las posiciones: los pesos cambian con los retornos, sin restaurarse diariamente. Cada retorno observado se utiliza una vez en su fecha; las probabilidades históricas no ponderan estos resultados.

Los retornos simples de rendimiento total incorporan la convención de reinversión implícita del proveedor. Esta es una contabilidad de rentabilidad, no una reconstrucción de títulos o flujos reales de dividendos; tampoco usamos precios ajustados como precios ejecutables. No incluimos costos.

Para cada instrumento $j$ y sesión $d$, iniciamos con $V_{j,0}=V_0w_{j,0}$ y aplicamos:

$$V_{j,d}=V_{j,d-1}(1+r_{j,d}),\qquad V_d=\sum_j V_{j,d},\qquad w_{j,d}=V_{j,d}/V_d,\qquad r_{p,d}=\sum_j w_{j,d-1}r_{j,d}.$$

El capital inicial ilustrativo es USD 1,000,000 por regla. El drawdown se mide frente al máximo acumulado incluyendo ese capital inicial.

In [3]:
# Copias previas para comprobar que la contabilidad no altera los objetivos ni la distribución.
saved_weights = {name: weights.copy() for name, weights in portfolios.items()}
saved_p_exp = p_exp.copy(); saved_q_vix = q_vix.copy()
saved_calendar = tuple(calendar); saved_rebalances = eligible_rebalances.clone()
initial_capital = 1_000_000.0
initial_execution = eligible_rebalances['fecha_ejecución'][0]
next_execution = eligible_rebalances['fecha_ejecución'][1]
first_return_date = eligible_rebalances['primer_retorno_nuevo'][0]
if initial_execution != date(2020, 1, 31) or first_return_date != date(2020, 2, 3):
    raise AssertionError('La primera ejecución o sesión atribuible no coincide con el calendario validado.')
if not (execution_date == initial_execution < first_return_date <= next_execution):
    raise AssertionError('Las fechas de ejecución y seguimiento no son coherentes.')

period = returns.filter((pl.col('session_date') > initial_execution) & (pl.col('session_date') <= next_execution))
period_dates = period['session_date'].to_list()
period_returns = period.select(SYMBOLS).to_numpy().astype(float)
expected_dates = calendar[positions[initial_execution] + 1:positions[next_execution] + 1]
if period_dates != expected_dates or not period_dates or period_dates[0] != first_return_date or period_dates[-1] != next_execution:
    raise AssertionError('Las fechas del periodo no coinciden con las sesiones publicadas.')
if period.columns != ['session_date', *SYMBOLS] or not np.isfinite(period_returns).all() or np.any(period_returns <= -1):
    raise AssertionError('Retornos o símbolos del periodo inválidos.')

# Cada fila de valores es un cierre; la fila cero es el instante inicial de la inversión.
accounting = {}
for name, target_weights in portfolios.items():
    instrument_values = np.empty((len(period_dates) + 1, len(SYMBOLS)))
    instrument_values[0] = initial_capital * target_weights
    daily_portfolio_returns = np.empty(len(period_dates))
    for d, asset_returns in enumerate(period_returns, start=1):
        previous_weights = instrument_values[d-1] / instrument_values[d-1].sum()
        daily_portfolio_returns[d-1] = previous_weights @ asset_returns
        instrument_values[d] = instrument_values[d-1] * (1.0 + asset_returns)
    capital = instrument_values.sum(axis=1)
    closing_weights = instrument_values / capital[:, None]
    running_peak = np.maximum.accumulate(capital)
    drawdown = capital / running_peak - 1.0
    accounting[name] = {'values': instrument_values, 'capital': capital, 'weights': closing_weights, 'daily_returns': daily_portfolio_returns, 'drawdown': drawdown}

summary = pl.DataFrame([{'regla': name, 'capital_final_usd': item['capital'][-1], 'retorno_acumulado_%': 100 * (item['capital'][-1] / initial_capital - 1), 'máximo_drawdown_%': 100 * item['drawdown'].min(), 'superó_15%_por_deriva': bool(np.any(item['weights'][1:] > 0.15 + 1e-12))} for name, item in accounting.items()])
weight_rows = []
for name, item in accounting.items():
    for j, symbol in enumerate(SYMBOLS):
        weight_rows.append({'regla': name, 'ETF': symbol, 'peso_inicial_%': 100 * saved_weights[name][j], 'peso_al_cierre_%': 100 * item['weights'][-1, j], 'superó_15%_durante_periodo': bool(np.any(item['weights'][1:, j] > 0.15 + 1e-12))})
period_weight_table = pl.DataFrame(weight_rows)

# Las identidades se contrastan sin redondear. En dólares usamos tolerancia absoluta 1e-7 y relativa 1e-12.
amount_atol, numeric_rtol, numeric_atol = 1e-7, 1e-12, 1e-12
period_controls = {
    'Mismas sesiones y retornos para las tres reglas': period_dates == expected_dates and period_returns.shape == (len(expected_dates), len(SYMBOLS)) and len(accounting) == len(portfolios) == 3,
    'Primer retorno posterior a ejecución': period_dates[0] == first_return_date > initial_execution,
    'Último retorno incluido antes de rebalancear': period_dates[-1] == next_execution == eligible_rebalances['fecha_ejecución'][1],
    'Pesos iniciales preservados': all(np.array_equal(item['values'][0], initial_capital * saved_weights[name]) and np.allclose(item['weights'][0], saved_weights[name], atol=numeric_atol, rtol=numeric_rtol) for name, item in accounting.items()),
    'Valores finitos y positivos; pesos suman uno': all(np.isfinite(item['values']).all() and np.all(item['values'] >= 0) and np.all(item['capital'] > 0) and np.isfinite(item['weights']).all() and np.allclose(item['weights'].sum(axis=1), 1, atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Retorno diario con pesos anteriores': all(np.allclose(item['daily_returns'], np.sum(item['weights'][:-1] * period_returns, axis=1), atol=numeric_atol, rtol=numeric_rtol) and np.allclose(item['capital'][1:] / item['capital'][:-1] - 1, item['daily_returns'], atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Valores por ETF concilian con producto de factores': all(np.allclose(item['values'][1:], initial_capital * saved_weights[name] * np.cumprod(1 + period_returns, axis=0), atol=amount_atol, rtol=numeric_rtol) for name, item in accounting.items()),
    'Capital concilia con composición diaria': all(np.allclose(item['capital'][1:], initial_capital * np.cumprod(1 + item['daily_returns']), atol=amount_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Drawdown incluye capital inicial': all(item['capital'][0] == initial_capital and item['drawdown'][0] == 0 and np.allclose(item['drawdown'], item['capital'] / np.maximum.accumulate(item['capital']) - 1, atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Objetivos, probabilidades y calendario preservados': all(np.array_equal(portfolios[name], weights) for name, weights in saved_weights.items()) and np.array_equal(p_exp, saved_p_exp) and np.array_equal(q_vix, saved_q_vix) and tuple(calendar) == saved_calendar and eligible_rebalances.equals(saved_rebalances),
}
if not all(period_controls.values()):
    raise AssertionError([name for name, ok in period_controls.items() if not ok])

print(f'Ejecución inicial: {initial_execution}; primera sesión: {period_dates[0]}; cierre siguiente rebalanceo incluido: {next_execution}; sesiones: {len(period_dates)}.')
display(summary)
with pl.Config(tbl_rows=30):
    display(period_weight_table)
capital_figure = go.Figure()
for name, item in accounting.items():
    capital_figure.add_scatter(x=[initial_execution, *period_dates], y=item['capital'], mode='lines', name=name, hovertemplate='%{x|%Y-%m-%d}<br>Capital: USD %{y:,.2f}<extra>%{fullData.name}</extra>')
capital_figure.update_layout(title='Capital bruto entre los dos primeros rebalanceos', xaxis_title='Fecha de cierre', yaxis_title='Capital (USD)', paper_bgcolor='#FFF8E7', plot_bgcolor='#FFF8E7', font_color='#2D2A26', legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0), margin=dict(l=80, r=40, t=110, b=60))
capital_figure.update_xaxes(gridcolor='#E7DDCC'); capital_figure.update_yaxes(gridcolor='#E7DDCC')
capital_figure.show()
display(pl.DataFrame({'Control': list(period_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in period_controls.values()]}))


Ejecución inicial: 2020-01-31; primera sesión: 2020-02-03; cierre siguiente rebalanceo incluido: 2020-02-28; sesiones: 19.


regla,capital_final_usd,retorno_acumulado_%,máximo_drawdown_%,superó_15%_por_deriva
str,f64,f64,f64,bool
"""Pesos iguales""",958645.234045,-4.135477,-6.759617,false
"""Mínimo CVaR exponencial""",966476.233418,-3.352377,-5.926497,true
"""Mínimo CVaR VIX""",963052.031754,-3.694797,-6.264971,true


regla,ETF,peso_inicial_%,peso_al_cierre_%,superó_15%_durante_periodo
str,str,f64,f64,bool
"""Pesos iguales""","""SPY""",12.5,12.006974,false
"""Pesos iguales""","""VEA""",12.5,12.041614,false
"""Pesos iguales""","""EEM""",12.5,12.546896,false
"""Pesos iguales""","""TLT""",12.5,13.903292,false
"""Pesos iguales""","""BIL""",12.5,13.054681,false
"""Pesos iguales""","""GLD""",12.5,12.956282,false
"""Pesos iguales""","""USO""",12.5,11.367229,false
"""Pesos iguales""","""VNQ""",12.5,12.123031,false
"""Mínimo CVaR exponencial""","""SPY""",15.0,14.291623,true


Control,Resultado
str,str
"""Mismas sesiones y retornos par…","""PASS"""
"""Primer retorno posterior a eje…","""PASS"""
"""Último retorno incluido antes …","""PASS"""
"""Pesos iniciales preservados""","""PASS"""
"""Valores finitos y positivos; p…","""PASS"""
"""Retorno diario con pesos anter…","""PASS"""
"""Valores por ETF concilian con …","""PASS"""
"""Capital concilia con composici…","""PASS"""
"""Drawdown incluye capital inici…","""PASS"""


Este periodo ilustra cómo se materializa una decisión y cómo derivan los pesos entre rebalanceos. Por sí solo no permite elegir un modelo ni demostrar superioridad. Los resultados son brutos y no incorporan costos.